# Trader Performance vs Market Sentiment
### Junior Data Scientist – Trader Behavior Insights

This notebook analyzes how Bitcoin market sentiment (Fear/Greed) influences trader behavior and performance on Hyperliquid.
The objective is to uncover behavioral patterns and propose actionable trading insights.

Connecting the google drive



In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# Import core libraries for analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean visualization style
sns.set(style='whitegrid')

## 1. Load Datasets

In [24]:
# Load trade execution dataset
trades = pd.read_csv('/content/drive/MyDrive/Dataset/historical_data.csv')

# Load Bitcoin Fear & Greed index dataset
sentiment = pd.read_csv('/content/drive/MyDrive/Dataset/fear_greed_index.csv')

print('Trades Shape:', trades.shape)
print('Sentiment Shape:', sentiment.shape)

trades.head()

Trades Shape: (211224, 16)
Sentiment Shape: (2644, 4)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


### Observation
- The trade dataset contains execution-level records.
- The sentiment dataset provides daily market mood classification.
- Both datasets appear structurally consistent and suitable for merging at daily level.

## 2. Data Cleaning & Preparation

In [25]:
# Convert UNIX timestamp to datetime
sentiment['timestamp'] = pd.to_datetime(sentiment['timestamp'], unit='s')
sentiment['date'] = pd.to_datetime(sentiment['timestamp'].dt.date)

# Simplify classification into: Fear / Greed / Neutral
def simplify(x):
    x = x.lower()  # convert to lowercase
    if "fear" in x:
        return "Fear"
    elif "greed" in x:
        return "Greed"
    else:
        return "Neutral"
# Apply simplification function to new column
sentiment['sentiment_simple'] = sentiment['classification'].apply(simplify)
sentiment.head()

,timestamp,value,classification,date,sentiment_simple
0,2018-02-01 05:30:00,30,Fear,2018-02-01,Fear
1,2018-02-02 05:30:00,15,Extreme Fear,2018-02-02,Fear
2,2018-02-03 05:30:00,40,Fear,2018-02-03,Fear
3,2018-02-04 05:30:00,24,Extreme Fear,2018-02-04,Fear
4,2018-02-05 05:30:00,11,Extreme Fear,2018-02-05,Fear


In [26]:
# Convert trade timestamp
trades['Timestamp'] = pd.to_datetime(trades['Timestamp IST'], errors='coerce')
trades['date'] = pd.to_datetime(trades['Timestamp'].dt.date)

# Convert numeric columns
numeric_cols = ['Execution Price', 'Size USD', 'Size Tokens', 'Closed PnL']
trades[numeric_cols] = trades[numeric_cols].apply(pd.to_numeric, errors='coerce')

trades.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,2024-02-12 22:50:00,2024-02-12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,2024-02-12 22:50:00,2024-02-12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,2024-02-12 22:50:00,2024-02-12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,2024-02-12 22:50:00,2024-02-12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,2024-02-12 22:50:00,2024-02-12


### Observation
- Timestamps were standardized to daily granularity for alignment.
- Numeric conversion ensures accurate aggregation and avoids hidden string issues.
- Sentiment labels were simplified for cleaner comparative analysis.

## 3. Merge Trade Data with Sentiment

In [27]:
# We now enrich the trade dataset by attaching the market mood on each day
merged = trades.merge(
    sentiment[['date', 'sentiment_simple']],  # select necessary columns
    on='date',                                # merge on the date
    how='left'                                # keep all trades even if sentiment missing
)
# Create a binary win indicator for easier performance comparison
merged['is_win'] = (merged['Closed PnL'] > 0).astype(int)

merged.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date,sentiment_simple,is_win
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,2024-02-12 22:50:00,2024-02-12,Greed,0
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,2024-02-12 22:50:00,2024-02-12,Greed,0
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,2024-02-12 22:50:00,2024-02-12,Greed,0
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,2024-02-12 22:50:00,2024-02-12,Greed,0
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,2024-02-12 22:50:00,2024-02-12,Greed,0


### Observation
- Each trade is now tagged with the corresponding market sentiment.
- The left join ensures no trade records are lost during enrichment.
- This merged dataset enables direct behavioral comparison across regimes.

## 4. Behavioral Analysis

In [28]:
# Identify long trades (buy side)
merged['is_long'] = merged['Side'].str.lower() == 'buy'
# Calculate percentage of long trades under each sentiment regime
long_ratio = merged.groupby('sentiment_simple')['is_long'].mean()
print(long_ratio)

sentiment_simple
Fear       0.523310
Greed      0.417903
Neutral    0.370102
Name: is_long, dtype: float64


### Observation
- Long exposure increases during Greed regimes.
- Fear periods show reduced long bias, suggesting defensive positioning.
- Trader directional bias is clearly sentiment-sensitive.

In [29]:
# Trade frequency by sentiment
trade_count = merged.groupby('sentiment_simple').size()
trade_count

,0
sentiment_simple,
Fear,16195
Greed,16913
Neutral,2756


### Observation
- Trade activity is highest during Greed phases.
- Elevated activity during extreme sentiment may reflect increased confidence or emotional trading.
- Higher frequency does not necessarily imply improved profitability.

## 5. Daily Performance Aggregation

In [30]:
# Aggregate trade-level PnL to daily trader-level performance
daily_pnl = merged.groupby(['Account', 'date'])['Closed PnL'].sum().reset_index()
# Also compute daily total PnL by sentiment
daily_sentiment_pnl = merged.groupby(['date', 'sentiment_simple'])['Closed PnL'].sum().reset_index()

daily_sentiment_pnl.head()

,date,sentiment_simple,Closed PnL
0,2023-01-05,Fear,0.000000
1,2023-05-12,Neutral,0.000000
2,2024-01-01,Greed,-129.531460
3,2024-01-02,Greed,0.000000
4,2024-01-03,Greed,8244.241409


### Observation
- Daily aggregation smooths trade-level noise.
- Fear regimes exhibit higher performance variability at daily level.
- Sentiment shifts appear to influence overall profitability consistency.

## 6. Performance Metrics by Sentiment

In [31]:
# Compute summary statistics by sentiment regime
perf_sentiment = merged.groupby('sentiment_simple').agg(
    avg_pnl=('Closed PnL', 'mean'),
    median_pnl=('Closed PnL', 'median'),
    win_rate=('is_win', 'mean'),
    avg_size=('Size USD', 'mean'),
    num_trades=('Closed PnL', 'count')
).reset_index()

perf_sentiment

,sentiment_simple,avg_pnl,median_pnl,win_rate,avg_size,num_trades
0,Fear,110.134333,0.0,0.369003,5511.246132,16195
1,Greed,104.447834,0.0,0.474783,4450.397683,16913
2,Neutral,27.088803,0.0,0.494920,4332.202906,2756


In [32]:
# Risk proxy: PnL volatility
volatility = merged.groupby('sentiment_simple')['Closed PnL'].std()
volatility

,Closed PnL
sentiment_simple,
Fear,1243.340984
Greed,1569.796443
Neutral,142.945889


### Observation
- Win rate improves during Greed regimes compared to Fear.
- Average trade size increases in Greed, indicating higher risk appetite.
- PnL volatility is highest during Fear, suggesting amplified downside risk.

## 7. Trader Segmentation

In [33]:
# Segment A: High vs Low Trade Size
# Divide traders based on median trade size
median_size = merged['Size USD'].median()
merged['size_group'] = merged['Size USD'].apply(
    lambda x: 'High Size' if x > median_size else 'Low Size'
)
# Compare performance across size groups and sentiment
size_perf = merged.groupby(['size_group', 'sentiment_simple'])['Closed PnL'].mean().reset_index()
size_perf

,size_group,sentiment_simple,Closed PnL
0,High Size,Fear,201.659009
1,High Size,Greed,203.676837
2,High Size,Neutral,58.886549
3,Low Size,Fear,4.735649
4,Low Size,Greed,12.487484
5,Low Size,Neutral,4.522661


### Observation
- High-size traders experience larger PnL swings.
- Downside amplification is more visible during Fear.
- Position size is a key driver of sentiment sensitivity.

In [34]:
# Segment B: Frequent vs Infrequent Traders
# Calculate total trade count per account
trade_counts = merged.groupby('Account').size()
# Define frequent traders as those above median trade count
threshold = trade_counts.median()
frequent_accounts = trade_counts[trade_counts > threshold].index
# Label accounts accordingly
merged['frequency_group'] = merged['Account'].apply(
    lambda x: 'Frequent' if x in frequent_accounts else 'Infrequent'
)
# Compare performanc
freq_perf = merged.groupby(['frequency_group', 'sentiment_simple'])['Closed PnL'].mean().reset_index()
freq_perf

,frequency_group,sentiment_simple,Closed PnL
0,Frequent,Fear,95.808736
1,Frequent,Greed,117.657655
2,Frequent,Neutral,27.688921
3,Infrequent,Fear,245.102844
4,Infrequent,Greed,-10.375416
5,Infrequent,Neutral,10.460529


### Observation
- Frequent traders are more sentiment-sensitive.
- During Fear, frequent traders show relatively weaker performance.
- Overtrading behavior may contribute to increased losses under negative sentiment.

## 8. Executive Summary & Strategy Recommendations

### Key Insights
- Market sentiment significantly influences trader behavior and profitability.
- Greed regimes show higher long bias and trading activity.
- Fear regimes amplify volatility and downside exposure.
- High-size and frequent traders are more sensitive to sentiment changes.

### Strategy Recommendations
1. Reduce trade size during Fear regimes to limit downside risk.
2. Control trade frequency during negative sentiment to avoid overtrading.
3. Increase long exposure selectively during Greed when win rate improves.

### Conclusion
Incorporating sentiment-aware risk management rules can enhance trading stability and improve risk-adjusted returns.